# Notebook 00 — Data Pipeline
## MIMIC-IV Demo → Processed Cohort

This notebook extracts ventilated patient data from the local MIMIC-IV Clinical Database Demo (v2.2),
preprocesses it, engineers features (including Mechanical Power via Gattinoni equation), and saves
a ready-to-use `processed_cohort.parquet` for downstream modelling notebooks (01–03).

**Run this notebook once** before running any strategy notebook.

In [ ]:
import sys, os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.utils.helpers import load_config
from src.data.extraction import LocalMIMICExtractor, get_extractor, MIMIC_ITEMIDS, MIMIC_LAB_ITEMIDS
from src.data.preprocessing import remove_outliers, impute_missing, resample_to_hourly
from src.features.engineering import (
    add_derived_features, calculate_mechanical_power,
    predicted_body_weight, get_state_feature_cols
)

config = load_config(os.path.join(PROJECT_ROOT, 'config', 'config.yaml'))
print(f"Project: {config['project']['name']} v{config['project']['version']}")
print(f"Data source: {config['data']['source']}")

## 1. Data Extraction from Local MIMIC-IV CSVs

In [ ]:
# Extract all tables using LocalMIMICExtractor
extractor = get_extractor(config)
raw_data = extractor.extract_all()

print("\n=== Extracted Tables ===")
for name, df in raw_data.items():
    print(f"  {name:20s}: {df.shape[0]:>6,} rows × {df.shape[1]} cols")

In [ ]:
episodes = raw_data['episodes']
print(f"\n=== Cohort Overview ===")
print(f"  Total ventilated stays:  {len(episodes)}")
print(f"  Mortality rate:          {episodes['hospital_expire_flag'].mean():.1%}")
print(f"  Median ventilation (h):  {episodes['vent_hours'].median():.1f}")
print(f"  Median age:              {episodes['age'].median():.0f}")
print(f"\nGender distribution:")
print(episodes['gender'].value_counts().to_string())

## 2. Pivot Chartevents to Wide Format

Map ITEMIDs to named columns and merge ventilator, vitals, and labs into a single wide DataFrame per (stay_id, charttime).

In [ ]:
# Build reverse ITEMID → column name mapping
itemid_to_col = {}
for col_name, ids in MIMIC_ITEMIDS.items():
    # Simplify names: tidal_volume_observed → tidal_volume, etc.
    simple = col_name.replace('_observed', '').replace('_set', '').replace('_total', '')
    for iid in ids:
        if iid not in itemid_to_col:  # prefer first (observed)
            itemid_to_col[iid] = simple

lab_itemid_to_col = {}
for col_name, ids in MIMIC_LAB_ITEMIDS.items():
    for iid in ids:
        lab_itemid_to_col[iid] = col_name

def pivot_long_to_wide(df_long, mapping):
    """Pivot long-format (stay_id, charttime, itemid, valuenum) to wide."""
    df = df_long.copy()
    df['col_name'] = df['itemid'].map(mapping)
    df = df.dropna(subset=['col_name'])
    # Take last value if duplicates
    df = df.drop_duplicates(subset=['stay_id', 'charttime', 'col_name'], keep='last')
    wide = df.pivot_table(index=['stay_id', 'charttime'], columns='col_name',
                          values='valuenum', aggfunc='last')
    wide = wide.reset_index()
    return wide

# Pivot ventilator and vitals (both come from chartevents)
vent_wide = pivot_long_to_wide(raw_data['ventilator'], itemid_to_col)
vitals_wide = pivot_long_to_wide(raw_data['vitals'], itemid_to_col)

# Pivot labs
labs_wide = pivot_long_to_wide(raw_data['labs'], lab_itemid_to_col)

print(f"Ventilator wide: {vent_wide.shape}")
print(f"Vitals wide:     {vitals_wide.shape}")
print(f"Labs wide:       {labs_wide.shape}")

In [ ]:
# Merge all time-series on (stay_id, charttime)
ts = vent_wide.merge(vitals_wide, on=['stay_id', 'charttime'], how='outer', suffixes=('', '_v'))
ts = ts.merge(labs_wide, on=['stay_id', 'charttime'], how='outer', suffixes=('', '_l'))

# Drop duplicate columns from merge
dup_cols = [c for c in ts.columns if c.endswith('_v') or c.endswith('_l')]
for dc in dup_cols:
    base = dc.rsplit('_', 1)[0]
    if base in ts.columns:
        ts[base] = ts[base].fillna(ts[dc])
    ts.drop(columns=[dc], inplace=True)

# Keep only rows from our cohort
stay_ids = episodes['stay_id'].unique()
ts = ts[ts['stay_id'].isin(stay_ids)].copy()
ts['charttime'] = pd.to_datetime(ts['charttime'])
ts = ts.sort_values(['stay_id', 'charttime']).reset_index(drop=True)

print(f"Combined time-series: {ts.shape}")
print(f"Columns: {sorted(ts.columns.tolist())}")

## 3. Preprocessing Pipeline

In [ ]:
# Resample to hourly resolution
ts_hourly = resample_to_hourly(ts, time_col='charttime')
print(f"Hourly time-series: {ts_hourly.shape}")
print(f"Stays in hourly data: {ts_hourly['stay_id'].nunique()}")

In [ ]:
# Remove physiological outliers
valid_ranges = config['preprocessing']['valid_ranges']
ts_clean = remove_outliers(ts_hourly, valid_ranges)
print("Outlier removal complete.")

In [ ]:
# Impute missing values
feat_cfg = config['features']
ts_imputed = impute_missing(
    ts_clean,
    strategies=config['preprocessing']['imputation'],
    vitals_cols=feat_cfg['dynamic_vitals'],
    labs_cols=feat_cfg['dynamic_labs'],
    vent_cols=feat_cfg['ventilator'],
    static_cols=feat_cfg['static'],
)
print("Imputation complete.")

In [ ]:
# Merge demographics
demo = raw_data['demographics']
outcomes = raw_data['outcomes']

ts_merged = ts_imputed.merge(demo, on='stay_id', how='left', suffixes=('', '_demo'))
ts_merged = ts_merged.merge(outcomes[['stay_id', 'hospital_expire_flag', 'icu_los_days']],
                            on='stay_id', how='left', suffixes=('', '_out'))

# Drop any suffix duplicates
for c in list(ts_merged.columns):
    if c.endswith('_demo') or c.endswith('_out'):
        ts_merged.drop(columns=[c], inplace=True, errors='ignore')

print(f"Merged shape: {ts_merged.shape}")

## 4. Feature Engineering

In [ ]:
# Compute predicted body weight
if 'height_cm' in ts_merged.columns and 'sex' in ts_merged.columns:
    ts_merged['predicted_body_weight'] = ts_merged.apply(
        lambda r: predicted_body_weight(r['height_cm'], r['sex'])
        if pd.notna(r.get('height_cm')) else np.nan,
        axis=1
    )

# Add derived features (MP, driving pressure, compliance, P/F ratio, TV/kg)
ts_final = add_derived_features(ts_merged)

# Check MP computation
mp_valid = ts_final['mechanical_power'].notna().sum() if 'mechanical_power' in ts_final.columns else 0
print(f"Mechanical Power computed for {mp_valid:,} hourly rows")
print(f"Stays with MP data: {ts_final[ts_final['mechanical_power'].notna()]['stay_id'].nunique()}")

In [ ]:
# Cohort summary
cohort_stays = ts_final['stay_id'].unique()
mp_stays = ts_final[ts_final['mechanical_power'].notna()]['stay_id'].unique()
outcomes_df = ts_final.groupby('stay_id')['hospital_expire_flag'].first()

print(f"\n{'='*50}")
print(f"COHORT SUMMARY")
print(f"{'='*50}")
print(f"Total stays in processed data:  {len(cohort_stays)}")
print(f"Stays with MP data:             {len(mp_stays)}")
print(f"Mortality rate:                 {outcomes_df.mean():.1%}")
print(f"  Survivors:                    {(outcomes_df == 0).sum()}")
print(f"  Non-survivors:                {(outcomes_df == 1).sum()}")

if 'mechanical_power' in ts_final.columns:
    mp_stats = ts_final['mechanical_power'].describe()
    print(f"\nMechanical Power (J/min):")
    print(f"  Mean ± SD:  {mp_stats['mean']:.1f} ± {mp_stats['std']:.1f}")
    print(f"  Median:     {mp_stats['50%']:.1f}")
    print(f"  Range:      [{mp_stats['min']:.1f}, {mp_stats['max']:.1f}]")

## 5. Data Quality Visualisation

In [ ]:
# Feature completeness heatmap (per-stay)
key_features = ['tidal_volume', 'respiratory_rate', 'peep', 'peak_pressure',
                'plateau_pressure', 'fio2', 'heart_rate', 'mean_arterial_pressure',
                'spo2', 'mechanical_power', 'driving_pressure', 'compliance']
key_features = [f for f in key_features if f in ts_final.columns]

completeness = ts_final.groupby('stay_id')[key_features].apply(
    lambda g: g.notna().mean()
).reset_index(drop=True) if len(key_features) > 0 else pd.DataFrame()

if not completeness.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(completeness.T, cmap='YlGnBu', vmin=0, vmax=1,
                xticklabels=False, yticklabels=True, ax=ax)
    ax.set_title('Feature Completeness per Stay (0 = all missing, 1 = fully available)')
    ax.set_xlabel('Patient Stay')
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'notebooks', 'figures', 'data_completeness.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Heatmap saved to notebooks/figures/data_completeness.png")

## 6. Save Processed Cohort

In [ ]:
# Save to parquet
output_dir = os.path.join(PROJECT_ROOT, config['data']['output_dir'])
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'processed_cohort.parquet')
ts_final.to_parquet(output_path, index=False)
print(f"Saved processed cohort: {output_path}")
print(f"Shape: {ts_final.shape}")

# Save metadata
meta = {
    'n_stays': int(ts_final['stay_id'].nunique()),
    'n_rows': int(len(ts_final)),
    'mortality_rate': float(outcomes_df.mean()),
    'n_survivors': int((outcomes_df == 0).sum()),
    'n_deaths': int((outcomes_df == 1).sum()),
    'columns': list(ts_final.columns),
    'mp_stays': int(len(mp_stays)),
}
meta_path = os.path.join(output_dir, 'cohort_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"Saved metadata: {meta_path}")

print(f"\n✓ Data pipeline complete. Ready for notebooks 01–03.")